# Frontend skill pilot: GPT-2 residuals and attention
Run this notebook in the ORCHARD Python environment after extraction. Inputs are research text, never executable instructions. Differences can reflect wording, length, and position; they do not demonstrate a safety mechanism.

The final token of each input is a different endpoint. Each chunk is processed independently and cannot attend to earlier chunks.

In [ ]:
from pathlib import Path
import json, re
import torch
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUN = ROOT / "results/frontend-pilot/with-attention"
manifest = json.loads((RUN / "manifest.json").read_text())
assert manifest.get("capture_attention"), "Run extraction with --capture-attention first"
summary = []
for record in manifest["files"]:
    for chunk in record["chunks"]:
        summary.append({"input": record["skill_file"], "chunk": chunk["chunk_index"],
                        "start": chunk["token_offset"],
                        "end_exclusive": chunk["token_offset"] + chunk["token_count"],
                        "total_input_tokens": record["token_count"]})
display(pd.DataFrame(summary))

In [ ]:
def layer_number(name):
    return int(re.search(r"(?:^|\.)blocks\.(\d+)\.", name).group(1))

samples = []
for record in manifest["files"]:
    chunk = record["chunks"][-1]
    saved = Path(chunk["output_file"])
    # Supports moving the run directory back from ORCHARD.
    path = RUN / saved.name
    payload = torch.load(path, map_location="cpu", weights_only=True)
    variant = Path(record["skill_file"]).parent.parent.name
    start, count = chunk["token_offset"], chunk["token_count"]
    label = f"{variant}: chunk {chunk['chunk_index']}, tokens [{start}, {start+count}), endpoint {start+count-1}"
    samples.append((label, payload))
assert len(samples) == 2, "Expected one benign and one adversarial input"

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for label, payload in samples:
    activations = payload["activations"]
    names = sorted((n for n in activations if n.endswith("hook_resid_post")), key=layer_number)
    assert names, "No residual post hooks"
    norms = [activations[n][0, -1].float().norm().item() for n in names]
    ax.plot([layer_number(n) for n in names], norms, marker="o", label=label)
ax.set(xlabel="Layer (zero based)", ylabel="Final-token residual L2 norm",
       title=f"{manifest['model_name']}: different input endpoints; final chunks only")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN / "residual_endpoint_norms.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(samples), 1, figsize=(16, 8), squeeze=False)
for ax, (label, payload) in zip(axes[:, 0], samples):
    names = sorted((n for n in payload["activations"] if n.endswith("hook_pattern")), key=layer_number)
    assert names, "No attention hooks"
    name = names[-1]
    pattern = payload["activations"][name].float()
    assert pattern.ndim == 4 and torch.isfinite(pattern).all()
    assert torch.allclose(pattern.sum(-1), torch.ones_like(pattern.sum(-1)), atol=.02, rtol=.02)
    weights = pattern[0, :, -1, :].mean(0)
    width = min(40, weights.numel())
    outside = weights[:-width].sum().item()
    tokens = [repr(t) for t in payload["tokens"][-width:]]
    ax.bar(range(width), weights[-width:].tolist())
    ax.set_xticks(range(width), tokens, rotation=90, fontsize=8)
    ax.set_ylabel("Mean attention across heads")
    ax.set_title(f"{manifest['model_name']} | layer {layer_number(name)} | {label}\nAttention outside displayed final {width} tokens within this chunk: {outside:.3%}; earlier chunks unavailable", fontsize=9)
fig.tight_layout()
fig.savefig(RUN / "final_token_attention.png", dpi=160)
plt.show()